# Human Activity Recognition with a Two-Stage Continuous HMM (TS-CHMM)
Reimplementation of **Ronao & Cho (2017)**, *Recognizing human activities from smartphone sensors using hierarchical continuous hidden Markov models*, on the **UCI HAR** dataset (Anguita et al., 2013).

This notebook produces every number, table and graph in the project report:

| Section | What it produces | Report |
|---|---|---|
| 1–2 | Data loading, class distribution, activity transition heat map | Table 4.1, Fig 4.1 |
| 3 | HMM core: training (Baum–Welch, hmmlearn) and a vectorised forward algorithm | Ch. 3 |
| 4 | Random Forest feature ranking (global, dynamic-only, static-only) | §3.6 |
| 5 | **Interpretation A** (feature-as-time): selection, TS-CHMM, BW sweep, flat baseline | §4.5, Figs 4.5–4.7, Table 4.4 |
| 6 | **Interpretation B** (window-as-time), corrected: selection, BW sweep, choice of L, final test | §4.6, Figs 4.8–4.9, Tables 4.5–4.6 |
| 7 | Comparison with the paper, per-class recall chart | §4.7, Table 4.7, Fig 4.10 |
| 8 | Window-by-window baselines: k-NN, softmax regression, Gaussian NB on raw / PCA / RF features | §4.4, Figs 4.2–4.4, Table 4.3 |
| 9 | LaTeX tables and a JSON summary of all results | — |

**Runtime.** With `QUICK = False` (the report's settings) a full run takes about 1.5–2 hours on 2 CPU cores, mostly Random Forests and cross-validation. Every heavy stage is cached in `cache_full/`, so re-running is fast. Set `QUICK = True` to check that everything runs in about 10 minutes, using smaller settings (the numbers will differ from the report).

In [ ]:
# !pip install -q hmmlearn      # uncomment on Colab

## 0. Configuration

In [ ]:
import os, io, json, time, zipfile, urllib.request, warnings
import numpy as np, pandas as pd, joblib
import matplotlib
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy.special import logsumexp
from hmmlearn import hmm
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
warnings.filterwarnings("ignore")
import logging; logging.getLogger("hmmlearn").setLevel(logging.ERROR)

QUICK = False                         # True = fast smoke test with smaller settings
DATA_DIR = os.environ.get("HAR_DATA_DIR", "UCI HAR Dataset")
ZIP_URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
CACHE = "cache_quick" if QUICK else "cache_full"
FIG, TAB = "figs", "tables"
for d in (CACHE, FIG, TAB): os.makedirs(d, exist_ok=True)

RF_SEEDS   = 5 if QUICK else 50       # number of Random Forests averaged for importance
RF_TREES   = 100
N_JOBS     = -1
MIN_COVAR  = 1e-3
RESTARTS   = 3                         # random restarts per HMM, best training log-likelihood kept

NAMES = ["W", "WU", "WD", "SI", "ST", "L"]
FULL_NAMES = ["Walking", "Walking Upstairs", "Walking Downstairs", "Sitting", "Standing", "Laying"]
DYN, STA = [1, 2, 3], [4, 5, 6]

def cached(name, fn):
    # Run fn() once and cache its result in CACHE/name.joblib
    path = os.path.join(CACHE, name + ".joblib")
    if os.path.exists(path):
        return joblib.load(path)
    t = time.time(); out = fn(); joblib.dump(out, path)
    print(f"  [{name}] computed in {time.time()-t:.0f}s")
    return out

# plotting style used for every figure in the report
C1, C2, C3, GREY, INK, INK2 = "#2a78d6", "#eb6834", "#1baf7a", "#8c8b86", "#0b0b0b", "#52514e"
plt.rcParams.update({"font.size": 10, "axes.edgecolor": "#b9b8b2", "axes.labelcolor": INK, "xtick.color": INK2,
    "ytick.color": INK2, "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8, "axes.spines.top": False,
    "axes.spines.right": False, "lines.linewidth": 2, "lines.markersize": 6, "legend.frameon": False, "savefig.dpi": 200})
def savefig(fig, name):
    fig.tight_layout(); fig.savefig(os.path.join(FIG, name)); plt.show(); plt.close(fig)

## 1. Data

In [ ]:
def get_data():
    if os.path.isdir(DATA_DIR):
        return
    print("Downloading UCI HAR ...")
    raw = urllib.request.urlopen(ZIP_URL).read()
    z = zipfile.ZipFile(io.BytesIO(raw))
    inner = [n for n in z.namelist() if n.endswith("UCI HAR Dataset.zip")]
    if inner:                                  # the UCI archive wraps the dataset zip in another zip
        z = zipfile.ZipFile(io.BytesIO(z.read(inner[0])))
    z.extractall(".")
get_data()

def load(split):
    d = os.path.join(DATA_DIR, split)
    X = np.loadtxt(os.path.join(d, f"X_{split}.txt"))
    y = np.loadtxt(os.path.join(d, f"y_{split}.txt"), dtype=int)
    s = np.loadtxt(os.path.join(d, f"subject_{split}.txt"), dtype=int)
    return X, y, s
X_train_raw, y_train, sub_train = load("train")
X_test_raw,  y_test,  sub_test  = load("test")
feature_names = [l.split(maxsplit=1)[1].strip() for l in open(os.path.join(DATA_DIR, "features.txt"))]

# z-score with training statistics only (paper eq. 20)
scaler = StandardScaler().fit(X_train_raw)
Xtr, Xte = scaler.transform(X_train_raw), scaler.transform(X_test_raw)
print("train", Xtr.shape, "test", Xte.shape, "| train subjects", np.unique(sub_train).size, "| test subjects", np.unique(sub_test).size)

## 2. Data analysis: class distribution (Table 4.1) and activity transition matrix (Fig 4.1)

In [ ]:
cnt_tr = np.bincount(y_train, minlength=7)[1:]; cnt_te = np.bincount(y_test, minlength=7)[1:]
table_classes = pd.DataFrame({"Activity": FULL_NAMES, "Group": ["Dynamic"]*3 + ["Static"]*3,
    "Train windows": cnt_tr, "Train share (%)": np.round(100*cnt_tr/cnt_tr.sum(), 1), "Test windows": cnt_te})
display(table_classes) if "display" in globals() else print(table_classes)

# maximum-likelihood first-order Markov chain over the training labels (rows are in recording order)
T = np.zeros((6, 6))
for a, b in zip(y_train[:-1], y_train[1:]): T[a-1, b-1] += 1
A_hat = T / T.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(5.4, 4.4))
im = ax.imshow(A_hat, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{A_hat[i,j]:.3f}", ha="center", va="center", fontsize=8, color="white" if A_hat[i,j] > .55 else INK)
ax.set_xticks(range(6), NAMES); ax.set_yticks(range(6), NAMES)
ax.set_xlabel("Next activity"); ax.set_ylabel("Current activity")
ax.set_title(f"Estimated transition matrix ({int(T.sum())} label pairs)", fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.046)
savefig(fig, "transition_heatmap.png")
print("self-transition probabilities:", np.round(np.diag(A_hat), 3))

## 3. HMM core
* `fit_hmm` trains a Gaussian HMM (diagonal covariance) with Baum–Welch via **hmmlearn**, from `RESTARTS` random starts (k-means initialisation of the means), keeping the start with the best training log-likelihood. `tol=0` so exactly `n_iter` iterations run.
* **Fix for length-1 sequences:** with $L=1$ no transition is ever observed and hmmlearn raises an error when re-estimating the transition matrix. The original notebook swallowed that error, so every model was `None` and every window got the first class. For $L=1$ the transition matrix is therefore not updated (`params="smc"`).
* `loglik` is the forward algorithm in log space, $\log P(O\mid\lambda)$, vectorised over many sequences at once. It gives the same values as `hmmlearn.score` (checked below) and is much faster than calling `score` once per sequence.

In [ ]:
def fit_hmm(blocks, n_states, n_iter, restarts=RESTARTS, seed=42, min_covar=MIN_COVAR):
    # blocks: array (n_sequences, L, K). Returns the best GaussianHMM or None.
    if blocks is None or len(blocks) == 0:
        return None
    n, L, K = blocks.shape
    flat, lengths = blocks.reshape(-1, K), [L] * n
    best, best_ll = None, -np.inf
    for r in range(seed, seed + restarts):
        m = hmm.GaussianHMM(n_components=n_states, covariance_type="diag", n_iter=n_iter, tol=0.0,
                            random_state=r, min_covar=min_covar, init_params="stmc",
                            params="smc" if L == 1 else "stmc")
        try:
            m.fit(flat, lengths); ll = m.score(flat, lengths)
        except Exception as e:
            print("  warning: HMM fit failed:", e); continue
        if np.isfinite(ll) and ll > best_ll:
            best, best_ll = m, ll
    return best

def loglik(m, blocks):
    # log P(O | lambda) for each sequence in blocks (n, L, K): forward algorithm in log space
    n, L, K = blocks.shape
    if m is None:
        return np.full(n, -np.inf)
    mu, var = m.means_, m._covars_                                   # (S, K) each
    logb = -0.5 * (np.log(2*np.pi*var)[None, None] + (blocks[:, :, None, :] - mu[None, None])**2 / var[None, None]).sum(-1)
    with np.errstate(divide="ignore"):
        logA, logpi = np.log(m.transmat_), np.log(m.startprob_)
    alpha = logpi[None] + logb[:, 0]                                 # alpha_1(j) = pi_j b_j(o_1)
    for t in range(1, L):                                            # alpha_t(j) = sum_i alpha_{t-1}(i) a_ij b_j(o_t)
        alpha = logsumexp(alpha[:, :, None] + logA[None], axis=1) + logb[:, t]
    return logsumexp(alpha, axis=1)                                  # P(O|lambda) = sum_i alpha_T(i)

def classify(models, blocks):
    # e* = argmax_e log P(O | lambda_e)
    keys = list(models)
    S = np.stack([loglik(models[k], blocks) for k in keys], 1)
    S = np.where(np.isnan(S), -np.inf, S)
    return np.array(keys)[np.argmax(S, 1)]

# sanity check against hmmlearn
_rng = np.random.default_rng(0); _b = _rng.normal(size=(20, 6, 4))
_m = fit_hmm(_b, 3, 5)
print("max |loglik - hmmlearn.score| =", max(abs(loglik(_m, _b[i:i+1])[0] - _m.score(_b[i])) for i in range(20)))

## 4. Random Forest feature ranking (paper §Feature selection using RF VI)

In [ ]:
def rf_importance(X, y, seeds=RF_SEEDS):
    def one(seed):
        return RandomForestClassifier(n_estimators=RF_TREES, random_state=seed, n_jobs=1).fit(X, y).feature_importances_
    return np.mean(Parallel(n_jobs=N_JOBS)(delayed(one)(s) for s in range(seeds)), axis=0)

def ranked_above_mean(imp):
    return [int(i) for i in np.argsort(imp)[::-1] if imp[i] > imp.mean()]

imp_all = cached("imp_all", lambda: rf_importance(Xtr, y_train))
imp_dyn = cached("imp_dyn", lambda: rf_importance(Xtr[y_train <= 3], y_train[y_train <= 3]))
imp_sta = cached("imp_sta", lambda: rf_importance(Xtr[y_train >= 4], y_train[y_train >= 4]))
rank_all, rank_dyn, rank_sta = ranked_above_mean(imp_all), ranked_above_mean(imp_dyn), ranked_above_mean(imp_sta)
print(f"features above mean importance: all {len(rank_all)}, dynamic {len(rank_dyn)}, static {len(rank_sta)} (of 561)")
print("top 5 overall:", [feature_names[i] for i in rank_all[:5]])

STEPS = [10, 70, len(rank_all)] if QUICK else sorted(set(list(range(10, len(rank_all), 20)) + [len(rank_all)]))
print("step-wise subset sizes k:", STEPS)

## 5. Interpretation A — feature-as-time (the original notebook's main model)
The $K$ selected features of **one window**, in importance order, form a length-$K$ sequence of scalars for a 1-D Gaussian HMM. Settings are the original notebook's: shuffled KFold (5 folds for Level 1, 3 for Level 2), Level 1 = 2 states / 2 BW iterations, Level 2 = 3 states / 10 iterations during selection and 15 in the final model.

In [ ]:
def seqA(X, feats):                     # (n, K) -> (n, K, 1): each window is one length-K scalar sequence
    return X[:, feats][:, :, None]

def a_cv_level1(feats, folds=5):
    errs = []
    yc = np.where(y_train <= 3, 0, 1)
    for tr, va in KFold(folds, shuffle=True, random_state=42).split(Xtr):
        m = {c: fit_hmm(seqA(Xtr[tr][yc[tr] == c], feats), 2, 2) for c in (0, 1)}
        errs.append(np.mean(classify(m, seqA(Xtr[va], feats)) != yc[va]))
    return float(np.mean(errs))

def a_cv_branch(feats, classes, folds=3, n_states=3, n_iter=10):
    mask = np.isin(y_train, classes); X, y = Xtr[mask], y_train[mask]
    errs = []
    for tr, va in KFold(folds, shuffle=True, random_state=42).split(X):
        m = {a: fit_hmm(seqA(X[tr][y[tr] == a], feats), n_states, n_iter) for a in classes}
        errs.append(np.mean(classify(m, seqA(X[va], feats)) != y[va]))
    return float(np.mean(errs))

class TS_CHMM_A:
    def __init__(self, F0, FD, FS, l2_states=3, l2_iters=15):
        self.F0, self.FD, self.FS, self.s, self.it = F0, FD, FS, l2_states, l2_iters
    def fit(self, X, y):
        yc = np.where(y <= 3, 0, 1)
        self.l1 = {c: fit_hmm(seqA(X[yc == c], self.F0), 2, 2) for c in (0, 1)}
        self.l2 = {a: fit_hmm(seqA(X[y == a], self.FD if a <= 3 else self.FS), self.s, self.it) for a in range(1, 7)}
        return self
    def predict(self, X):
        route = classify(self.l1, seqA(X, self.F0)); p = np.zeros(len(X), int)
        d = route == 0
        p[d]  = classify({a: self.l2[a] for a in DYN}, seqA(X[d],  self.FD))
        p[~d] = classify({a: self.l2[a] for a in STA}, seqA(X[~d], self.FS))
        return p

A = {}
A["f0_err"] = cached("A_f0_err", lambda: [a_cv_level1(rank_all[:k]) for k in STEPS])
F0_A = rank_all[:STEPS[int(np.argmin(A["f0_err"]))]]
A["fd_err"] = cached("A_fd_err", lambda: [a_cv_branch(rank_all[:k], DYN) for k in STEPS])
A["fs_err"] = cached("A_fs_err", lambda: [a_cv_branch(rank_all[:k], STA) for k in STEPS])
FD_A = rank_all[:STEPS[int(np.argmin(A["fd_err"]))]]
FS_A = rank_all[:STEPS[int(np.argmin(A["fs_err"]))]]
print(f"Interpretation A: |F0|={len(F0_A)}, |FD|={len(FD_A)}, |FS|={len(FS_A)}")
for k, e0, ed, es in zip(STEPS, A["f0_err"], A["fd_err"], A["fs_err"]):
    print(f"  k={k:3d}  L1 err={e0:.4f}  dyn err={ed:.4f}  sta err={es:.4f}")

A["pred"] = cached("A_pred", lambda: TS_CHMM_A(F0_A, FD_A, FS_A).fit(Xtr, y_train).predict(Xte))
print(f"\nTS-CHMM (A) test accuracy: {accuracy_score(y_test, A['pred']):.4f}")

In [ ]:
# Level-2 Baum-Welch iteration sweep (test accuracy per branch, for analysis only - not used to choose anything)
A_ITERS = [1, 10] if QUICK else [1, 5, 10, 27, 40, 58, 70]
def a_sweep():
    dyn, sta = [], []
    for it in A_ITERS:
        p = TS_CHMM_A(F0_A, FD_A, FS_A, l2_iters=it).fit(Xtr, y_train).predict(Xte)
        dyn.append(accuracy_score(y_test[y_test <= 3], p[y_test <= 3])); sta.append(accuracy_score(y_test[y_test >= 4], p[y_test >= 4]))
        print(f"  iters={it:2d}  dynamic={dyn[-1]:.4f}  static={sta[-1]:.4f}")
    return dyn, sta
A["sweep_dyn"], A["sweep_sta"] = cached("A_sweep", a_sweep)

# flat 6-class CHMM baseline on F0 (3 states, 30 iterations)
def a_flat():
    m = {a: fit_hmm(seqA(Xtr[y_train == a], F0_A), 3, 30) for a in range(1, 7)}
    return classify(m, seqA(Xte, F0_A))
A["flat_pred"] = cached("A_flat", a_flat)
print(f"Flat CHMM (A) test accuracy: {accuracy_score(y_test, A['flat_pred']):.4f}")

In [ ]:
# Figures 4.5 - 4.7
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(STEPS, np.array(A["f0_err"])*100, "-o", color=C1)
ax.axvline(len(F0_A), color=GREY, ls="--", lw=1); ax.text(len(F0_A)+2, max(A["f0_err"])*85, f"chosen k = {len(F0_A)}", color=INK2)
ax.set_xlabel("Number of top-ranked features k"); ax.set_ylabel("Level-1 CV error (%)")
savefig(fig, "A_f0_error.png")

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(STEPS, np.array(A["fd_err"])*100, "-o", color=C1, label="Dynamic branch (W/WU/WD)")
ax.plot(STEPS, np.array(A["fs_err"])*100, "-s", color=C2, label="Static branch (SI/ST/L)")
ax.set_xlabel("Number of top-ranked features k"); ax.set_ylabel("Level-2 CV error (%)"); ax.legend()
savefig(fig, "A_l2_error.png")

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(A_ITERS, np.array(A["sweep_dyn"])*100, "-o", color=C1, label="Dynamic branch")
ax.plot(A_ITERS, np.array(A["sweep_sta"])*100, "-s", color=C2, label="Static branch")
ax.set_xlabel("Baum–Welch iterations (Level 2, 3 states)"); ax.set_ylabel("Test accuracy (%)"); ax.legend()
savefig(fig, "A_bw_sweep.png")

## 6. Interpretation B — window-as-time (corrected)
Each window is one $K$-dimensional observation, and a sequence is $L$ consecutive windows of the same subject.

Fixes compared with the original notebook:
1. **Training blocks** are cut from runs of consecutive *original* rows, so filtering to one branch no longer glues together runs that were separated in time.
2. **Test/validation context** for window $i$ is that subject's last $L$ windows in the full time-ordered stream (no labels, no future windows). It is no longer built only from windows already routed to a branch, or from windows of the true branch.
3. **Every setting** (features, iterations, $L$) is chosen by **GroupKFold by subject** on the training subjects. The test set is used once, at the end. The original picked $L$ by test accuracy.
4. **Length-1 sequences** train correctly (see `fit_hmm`).

In [ ]:
def runs_of(mask):
    idx = np.where(mask)[0]
    return [] if len(idx) == 0 else np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)

def train_blocks(X, y, subj, rows_mask, label_of_row, feats, L):
    # {label: array (n_blocks, L, K)} of L consecutive ORIGINAL rows with the same subject and label
    out = {}
    for s in np.unique(subj):
        for lab in np.unique(label_of_row[rows_mask & (subj == s)]):
            for run in runs_of(rows_mask & (subj == s) & (label_of_row == lab)):
                for i in range(len(run) - L + 1):
                    out.setdefault(lab, []).append(X[run[i:i+L]][:, feats])
    return {k: np.array(v) for k, v in out.items()}

def context_index(subj, L):
    # row i -> indices of the last L rows of the same subject ending at i (padded with the first one)
    ctx, hist = np.zeros((len(subj), L), dtype=int), {}
    for i, s in enumerate(subj):
        h = hist.setdefault(s, []); h.append(i); w = h[-L:]
        ctx[i] = [w[0]] * (L - len(w)) + w
    return ctx

class TS_CHMM_B:
    def __init__(self, F0, FD, FS, params, L):
        # params: {activity: (n_states, n_iter)}
        self.F0, self.FD, self.FS, self.params, self.L = F0, FD, FS, params, L
    def fit(self, X, y, subj):
        L, yc = self.L, np.where(y <= 3, 0, 1)
        b0 = train_blocks(X, y, subj, np.ones(len(y), bool), yc, self.F0, L)
        self.l1 = {c: fit_hmm(b0.get(c), 2, 2) for c in (0, 1)}               # Level 1: 2 states, 2 BW iterations
        bd = train_blocks(X, y, subj, y <= 3, y, self.FD, L)
        bs = train_blocks(X, y, subj, y >= 4, y, self.FS, L)
        self.l2 = {a: fit_hmm((bd if a <= 3 else bs).get(a), *self.params[a]) for a in range(1, 7)}
        return self
    def predict(self, X, subj, return_route=False):
        ctx = context_index(subj, self.L)
        route = classify(self.l1, X[ctx][:, :, self.F0]); p = np.zeros(len(X), int); d = route == 0
        if d.any():    p[d]  = classify({a: self.l2[a] for a in DYN}, X[ctx[d]][:, :, self.FD])
        if (~d).any(): p[~d] = classify({a: self.l2[a] for a in STA}, X[ctx[~d]][:, :, self.FS])
        return (p, route) if return_route else p

class Flat_CHMM_B:
    def __init__(self, F, L, n_states=3, n_iter=15): self.F, self.L, self.s, self.it = F, L, n_states, n_iter
    def fit(self, X, y, subj):
        b = train_blocks(X, y, subj, np.ones(len(y), bool), y, self.F, self.L)
        self.m = {a: fit_hmm(b.get(a), self.s, self.it) for a in range(1, 7)}; return self
    def predict(self, X, subj):
        ctx = context_index(subj, self.L); return classify(self.m, X[ctx][:, :, self.F])

FOLDS = list(GroupKFold(n_splits=5).split(Xtr, y_train, sub_train))

def b_cv_level1(F, L):
    errs = []
    for tr, va in FOLDS:
        yc = np.where(y_train[tr] <= 3, 0, 1)
        b = train_blocks(Xtr[tr], y_train[tr], sub_train[tr], np.ones(len(tr), bool), yc, F, L)
        m = {c: fit_hmm(b.get(c), 2, 2) for c in (0, 1)}
        ctx = context_index(sub_train[va], L)
        errs.append(np.mean(classify(m, Xtr[va][ctx][:, :, F]) != np.where(y_train[va] <= 3, 0, 1)))
    return float(np.mean(errs))

def b_cv_branch(F, classes, L, params):
    # branch accuracy assuming correct routing; context from the full validation stream
    corr = tot = 0
    for tr, va in FOLDS:
        b = train_blocks(Xtr[tr], y_train[tr], sub_train[tr], np.isin(y_train[tr], classes), y_train[tr], F, L)
        m = {a: fit_hmm(b.get(a), *params[a]) for a in classes}
        ctx = context_index(sub_train[va], L); keep = np.isin(y_train[va], classes)
        corr += np.sum(classify(m, Xtr[va][ctx[keep]][:, :, F]) == y_train[va][keep]); tot += keep.sum()
    return corr / tot

def b_cv_full(F0, FD, FS, params, L):
    corr = 0
    for tr, va in FOLDS:
        p = TS_CHMM_B(F0, FD, FS, params, L).fit(Xtr[tr], y_train[tr], sub_train[tr]).predict(Xtr[va], sub_train[va])
        corr += np.sum(p == y_train[va])
    return corr / len(y_train)

In [ ]:
# 6a. step-wise feature selection (L = 5, 3 states, 10 iterations)
L_SEL = 5
default = {a: (3, 10) for a in range(1, 7)}
B = {}
B["f0_err"] = cached("B_f0_err", lambda: [b_cv_level1(rank_all[:k], L_SEL) for k in STEPS])
B["fd_acc"] = cached("B_fd_acc", lambda: [b_cv_branch(rank_dyn[:k], DYN, L_SEL, default) for k in STEPS])
B["fs_acc"] = cached("B_fs_acc", lambda: [b_cv_branch(rank_sta[:k], STA, L_SEL, default) for k in STEPS])
F0_B = rank_all[:STEPS[int(np.argmin(B["f0_err"]))]]
FD_B = rank_dyn[:STEPS[int(np.argmax(B["fd_acc"]))]]
FS_B = rank_sta[:STEPS[int(np.argmax(B["fs_acc"]))]]
print(f"Interpretation B: |F0|={len(F0_B)}, |FD|={len(FD_B)}, |FS|={len(FS_B)}")
for k, e0, ad, as_ in zip(STEPS, B["f0_err"], B["fd_acc"], B["fs_acc"]):
    print(f"  k={k:3d}  L1 CV err={e0:.4f}  dyn CV acc={ad:.4f}  sta CV acc={as_:.4f}")

In [ ]:
# 6b. Level-2 Baum-Welch iteration sweep (3 states, L = 5). CV picks the peak; test accuracy recorded for the plot only.
#     The report stopped at 27 (dynamic) and 20 (static) because CV accuracy was already falling; extend the lists if you like.
B_ITERS = {"dyn": [1, 5] if QUICK else [1, 5, 10, 20, 27], "sta": [1, 5] if QUICK else [1, 5, 10, 20]}
def branch_test(F, classes, params, L):
    b = train_blocks(Xtr, y_train, sub_train, np.isin(y_train, classes), y_train, F, L)
    m = {a: fit_hmm(b.get(a), *params[a]) for a in classes}
    ctx = context_index(sub_test, L); keep = np.isin(y_test, classes)
    return float(np.mean(classify(m, Xte[ctx[keep]][:, :, F]) == y_test[keep]))
def b_sweep(F, classes, iters):
    cv, te = [], []
    for it in iters:
        p = {a: (3, it) for a in classes}
        cv.append(b_cv_branch(F, classes, L_SEL, p)); te.append(branch_test(F, classes, p, L_SEL))
        print(f"  iters={it:2d}  CV={cv[-1]:.4f}  test={te[-1]:.4f}")
    return cv, te
B["sweep_dyn"] = cached("B_sweep_dyn", lambda: b_sweep(FD_B, DYN, B_ITERS["dyn"]))
B["sweep_sta"] = cached("B_sweep_sta", lambda: b_sweep(FS_B, STA, B_ITERS["sta"]))
peak = {n: B_ITERS[n][int(np.argmax(B[f"sweep_{n}"][0]))] for n in ("dyn", "sta")}
params_B = {a: (3, peak["dyn"]) for a in DYN}; params_B.update({a: (3, peak["sta"]) for a in STA})
print("CV-peak iterations:", peak)

In [ ]:
# 6c. choose the sequence length L by CV of the whole two-stage model, then evaluate every L on test (analysis)
LS = [1, 3] if QUICK else [1, 3, 5, 10]
def b_final(L):
    cv = b_cv_full(F0_B, FD_B, FS_B, params_B, L)
    p, route = TS_CHMM_B(F0_B, FD_B, FS_B, params_B, L).fit(Xtr, y_train, sub_train).predict(Xte, sub_test, return_route=True)
    flat = Flat_CHMM_B(F0_B, L).fit(Xtr, y_train, sub_train).predict(Xte, sub_test)
    print(f"  L={L:2d}  CV={cv:.4f}  test={accuracy_score(y_test, p):.4f}  flat test={accuracy_score(y_test, flat):.4f}")
    return dict(cv=cv, pred=p, route=route, flat=flat)
B["byL"] = {L: cached(f"B_final_L{L}", lambda L=L: b_final(L)) for L in LS}
best_L = LS[int(np.argmax([B["byL"][L]["cv"] for L in LS]))]
B["pred"], B["flat_pred"], B["route"] = B["byL"][best_L]["pred"], B["byL"][best_L]["flat"], B["byL"][best_L]["route"]
print(f"\nL chosen by CV: {best_L}   ->   TS-CHMM (B) test accuracy {accuracy_score(y_test, B['pred']):.4f}")

In [ ]:
# Figures 4.8 - 4.9
fig, axs = plt.subplots(1, 2, figsize=(10, 3.3))
axs[0].plot(STEPS, np.array(B["f0_err"])*100, "-o", color=C1); axs[0].axvline(len(F0_B), color=GREY, ls="--", lw=1)
axs[0].set_title("Level 1 (dynamic vs static)", fontsize=10); axs[0].set_xlabel("Number of ranked features k"); axs[0].set_ylabel("CV error (%)")
axs[1].plot(STEPS, np.array(B["fd_acc"])*100, "-o", color=C1, label="Dynamic branch")
axs[1].plot(STEPS, np.array(B["fs_acc"])*100, "-s", color=C2, label="Static branch")
axs[1].set_title("Level 2 (within branch)", fontsize=10); axs[1].set_xlabel("Number of ranked features k"); axs[1].set_ylabel("CV accuracy (%)"); axs[1].legend()
savefig(fig, "B_selection.png")

fig, axs = plt.subplots(1, 2, figsize=(10, 3.3), sharey=True)
for ax, n, t in [(axs[0], "dyn", "Dynamic branch (W/WU/WD)"), (axs[1], "sta", "Static branch (SI/ST/L)")]:
    cv, te = B[f"sweep_{n}"]
    ax.plot(B_ITERS[n], np.array(cv)*100, "-o", color=C1, label="CV (training subjects)")
    ax.plot(B_ITERS[n], np.array(te)*100, "-s", color=C2, label="Test subjects")
    ax.axvline(peak[n], color=GREY, ls="--", lw=1); ax.set_title(t, fontsize=10); ax.set_xlabel("Baum–Welch iterations")
axs[0].set_ylabel("Branch accuracy (%)"); axs[0].legend()
savefig(fig, "B_bw_sweep.png")

table_L = pd.DataFrame({"L (windows)": LS, "Context (s)": [1.28*(L-1) + 2.56 for L in LS],
    "CV accuracy (%)": [100*B["byL"][L]["cv"] for L in LS],
    "Test accuracy (%)": [100*accuracy_score(y_test, B["byL"][L]["pred"]) for L in LS],
    "Flat CHMM test (%)": [100*accuracy_score(y_test, B["byL"][L]["flat"]) for L in LS]}).round(2)
print(table_L.to_string(index=False))

## 7. Results: confusion matrices, branch accuracies, comparison with the paper (Tables 4.4, 4.6, 4.7; Fig 4.10)

In [ ]:
def summary(pred, route=None):
    yc = np.where(y_test <= 3, 0, 1)
    rc = route if route is not None else np.where(pred <= 3, 0, 1)
    return {"Level 1": accuracy_score(yc, rc), "Dynamic branch": accuracy_score(y_test[y_test <= 3], pred[y_test <= 3]),
            "Static branch": accuracy_score(y_test[y_test >= 4], pred[y_test >= 4]), "Overall": accuracy_score(y_test, pred)}

def cm_plot(pred, title, name):
    cm = confusion_matrix(y_test, pred); rec = cm / cm.sum(1, keepdims=True)
    fig, ax = plt.subplots(figsize=(4.8, 4.0)); ax.imshow(rec, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
    for i in range(6):
        for j in range(6):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9, color="white" if rec[i, j] > .55 else INK)
    ax.set_xticks(range(6), NAMES); ax.set_yticks(range(6), NAMES); ax.set_xlabel("Predicted activity"); ax.set_ylabel("True activity")
    ax.set_title(title, fontsize=10); [s.set_visible(False) for s in ax.spines.values()]
    savefig(fig, name)

for tag, pred in [("A", A["pred"]), ("B", B["pred"])]:
    print(f"===== Interpretation {tag} =====")
    print(classification_report(y_test, pred, target_names=NAMES, digits=4))
    print(confusion_matrix(y_test, pred))
cm_plot(A["pred"], "Interpretation A, TS-CHMM", "A_cm.png")
cm_plot(B["pred"], f"Interpretation B (L={best_L})", "B_cm.png")

PAPER = {"Level 1": 1.0, "Dynamic branch": .9244, "Static branch": .9338, "Overall": .9318}   # Ronao & Cho (2017), reported
PAPER_RECALL = [95.77, 95.75, 83.57, 93.48, 87.41, 100.00]                                    # paper Table 3
comparison = pd.DataFrame({"Ronao & Cho (reported)": PAPER, "Ours, Interpretation A": summary(A["pred"]),
                           f"Ours, Interpretation B (L={best_L})": summary(B["pred"], B["route"])}).T * 100
comparison["Overall"] = comparison["Overall"]
flat_rows = pd.DataFrame({"Overall": [100*accuracy_score(y_test, A["flat_pred"]), 100*accuracy_score(y_test, B["flat_pred"])]},
                         index=["Flat CHMM, Interpretation A", "Flat CHMM, Interpretation B"])
comparison = pd.concat([comparison, flat_rows]).round(2)
print(comparison.to_string())

recA = 100 * np.diag(confusion_matrix(y_test, A["pred"])) / np.bincount(y_test)[1:]
recB = 100 * np.diag(confusion_matrix(y_test, B["pred"])) / np.bincount(y_test)[1:]
x, w = np.arange(6), .27
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(x - w, PAPER_RECALL, w*.92, color=GREY, label="Ronao & Cho (reported)")
ax.bar(x, recA, w*.92, color=C1, label="Ours, Interpretation A")
ax.bar(x + w, recB, w*.92, color=C2, label=f"Ours, Interpretation B (L={best_L})")
ax.set_xticks(x, NAMES); ax.set_ylabel("Recall (%)"); ax.set_ylim(0, 105); ax.grid(axis="x", visible=False)
ax.legend(loc="upper center", bbox_to_anchor=(.5, -.14), ncol=3)
savefig(fig, "recall_compare.png")

## 8. Window-by-window baselines (Figs 4.2–4.4, Table 4.3)
These use the original (un-standardised) features, as in the report's baseline experiments.

In [ ]:
Xr, Xs = X_train_raw, X_test_raw
def knn_curve(a, b):
    return [KNeighborsClassifier(k).fit(a, y_train).score(b, y_test) for k in range(1, 11)]
pca95 = PCA(0.95).fit(Xr)
knn_raw = cached("bl_knn_raw", lambda: knn_curve(Xr, Xs))
knn_pca = cached("bl_knn_pca", lambda: knn_curve(pca95.transform(Xr), pca95.transform(Xs)))
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(range(1, 11), knn_raw, "-o", color=C1, label="Original features (561 dims)")
ax.plot(range(1, 11), knn_pca, "--s", color=C2, label=f"PCA-reduced features ({pca95.n_components_} dims)")
ax.set_xticks(range(1, 11)); ax.set_xlabel("Number of neighbours (k)"); ax.set_ylabel("Test accuracy"); ax.legend()
ax.set_title("k-NN test accuracy: original vs PCA-reduced features", fontsize=10)
savefig(fig, "knn_orig_vs_pca.png")
print("best k-NN (raw):", max(knn_raw), "at k =", 1 + int(np.argmax(knn_raw)))

MODELS = {"Softmax Regression": lambda: LogisticRegression(max_iter=2000), "K-NN (k=6)": lambda: KNeighborsClassifier(6),
          "Gaussian Naive Bayes": lambda: GaussianNB()}
COLORS = [C1, C2, C3]
VARS = [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.99]
def pca_sweep():
    out = {m: [] for m in MODELS}; ncomp = []
    for v in VARS:
        p = PCA(v).fit(Xr); a, b = p.transform(Xr), p.transform(Xs); ncomp.append(p.n_components_)
        for m, f in MODELS.items(): out[m].append(f().fit(a, y_train).score(b, y_test))
    return out, ncomp
pca_res, ncomp = cached("bl_pca_sweep", pca_sweep)
fig, ax = plt.subplots(figsize=(7, 3.6))
for (m, v), c in zip(pca_res.items(), COLORS): ax.plot(range(len(VARS)), v, "-o", color=c, label=m)
for i, n in enumerate(ncomp): ax.annotate(f"n={n}", (i, max(r[i] for r in pca_res.values())), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=8, color=INK2)
ax.set_xticks(range(len(VARS)), [f"{v:.2f}" for v in VARS]); ax.set_xlabel("PCA explained-variance threshold"); ax.set_ylabel("Test accuracy"); ax.legend()
ax.set_title("Test accuracy vs PCA retained variance", fontsize=10)
savefig(fig, "pca_variance_sweep.png")

M_RANGE = list(range(2, len(rank_all) - 1, 32 if QUICK else 1))
def rf_sweep():
    out = {m: [] for m in MODELS}
    for mm in M_RANGE:
        f_ = rank_all[:mm]
        for m, f in MODELS.items(): out[m].append(f().fit(Xr[:, f_], y_train).score(Xs[:, f_], y_test))
    return out
rf_res = cached("bl_rf_sweep", rf_sweep)
fig, ax = plt.subplots(figsize=(7, 3.6))
for (m, v), c in zip(rf_res.items(), COLORS): ax.plot(M_RANGE, v, color=c, label=m)
ax.set_xlabel("Number of top features (m)"); ax.set_ylabel("Test accuracy"); ax.legend()
ax.set_title("Test accuracy vs number of RF-selected features (m)", fontsize=10)
savefig(fig, "rf_feature_sweep.png")

def base_table():
    rows = {}
    rows["All 561 features"] = [LogisticRegression(max_iter=2000).fit(Xr, y_train).score(Xs, y_test), KNeighborsClassifier(6).fit(Xr, y_train).score(Xs, y_test), np.nan]
    for v in (0.95, 0.99):
        p = PCA(v).fit(Xr); a, b = p.transform(Xr), p.transform(Xs)
        rows[f"PCA {int(v*100)}% variance ({p.n_components_} comp.)"] = [f().fit(a, y_train).score(b, y_test) for f in MODELS.values()]
    for mm in (70, 129):
        f_ = rank_all[:mm]; rows[f"Top {mm} RF features"] = [f().fit(Xr[:, f_], y_train).score(Xs[:, f_], y_test) for f in MODELS.values()]
    return pd.DataFrame(rows, index=["Softmax regression", "k-NN (k=6)", "Gaussian NB"]).T * 100
table_base = cached("bl_table", base_table).round(2)
print(table_base.to_string())

## 9. Export: LaTeX tables and a JSON summary

In [ ]:
def latex_cm(pred, label, caption):
    cm = confusion_matrix(y_test, pred); cr = classification_report(y_test, pred, target_names=NAMES, output_dict=True)
    rows = [f"{n} & " + " & ".join(map(str, cm[i])) + f" & {100*cr[n]['recall']:.2f}\\\\" for i, n in enumerate(NAMES)]
    prec = " & ".join(f"{100*cr[n]['precision']:.2f}" for n in NAMES)
    return ("\\begin{table}[H]\n\\centering\n\\caption{" + caption + "}\n\\label{" + label + "}\n"
            "\\begin{tabular}{l|cccccc|c}\n\\toprule\nTrue $\\backslash$ Predicted & W & WU & WD & SI & ST & L & Recall (\\%)\\\\\n\\midrule\n"
            + "\n".join(rows) + f"\n\\midrule\nPrecision (\\%) & {prec} & {100*cr['accuracy']:.2f}\\\\\n\\bottomrule\n\\end{{tabular}}\n\\end{{table}}\n")
open(os.path.join(TAB, "A_confusion.tex"), "w").write(latex_cm(A["pred"], "tab:A_cm", "Interpretation A: confusion matrix on the 2947 test windows."))
open(os.path.join(TAB, "B_confusion.tex"), "w").write(latex_cm(B["pred"], "tab:B_cm", f"Interpretation B ($L={best_L}$): confusion matrix on the 2947 test windows."))
for name, df in [("comparison", comparison), ("sequence_length", table_L), ("baselines", table_base), ("classes", table_classes)]:
    df.to_latex(os.path.join(TAB, f"{name}.tex"), float_format="%.2f", na_rep="--", index=(name != "classes" and name != "sequence_length"))

results = {"A": {"F0": len(F0_A), "FD": len(FD_A), "FS": len(FS_A), **{k: round(100*v, 2) for k, v in summary(A["pred"]).items()},
                 "flat": round(100*accuracy_score(y_test, A["flat_pred"]), 2)},
           "B": {"F0": len(F0_B), "FD": len(FD_B), "FS": len(FS_B), "peak_iters": peak, "L": best_L,
                 **{k: round(100*v, 2) for k, v in summary(B["pred"], B["route"]).items()},
                 "flat": round(100*accuracy_score(y_test, B["flat_pred"]), 2)},
           "paper": {k: round(100*v, 2) for k, v in PAPER.items()}}
json.dump(results, open("results_summary.json", "w"), indent=2)
print(json.dumps(results, indent=2))
print("figures ->", FIG, "| tables ->", TAB)